In [0]:
from pyspark.sql.functions import col
from pyspark.sql import Row


df2=spark.read.table("sales_project_db.bronze_sales")

total_rows= df2.count()

null_orders = df2.filter(col("Order_ID").isNull()).count()

null_customers = df2.filter(col("Customer_ID").isNull()).count()

duplicates = df2.groupBy("Order_ID","Product_ID").count().filter("count > 1").count()

negative_values = df2.filter("Sales <= 0 OR Quantity <= 0").count()

invalid_discount = df2.filter("Discount < 0 OR Discount > 1").count()


report_data_bronze = [
    Row(check="total_rows", value=total_rows, status="INFO"),
    Row(check="null_order_id", value=null_orders, status="PASS" if null_orders == 0 else "FAIL"),
    Row(check="null_customer_id", value=null_customers, status="PASS" if null_customers == 0 else "FAIL"),
    Row(check="duplicate_orders", value=duplicates, status="PASS" if duplicates == 0 else "FAIL"),
    Row(check="negative_values", value=negative_values, status="PASS" if negative_values == 0 else "FAIL"),
]

report_df2 = spark.createDataFrame(report_data_bronze)

display(report_df2)

In [0]:
from pyspark.sql.functions import col

df=spark.read.table("sales_project_db.silver_sales")

total_rows= df.count()

null_orders = df.filter(col("Order_ID").isNull()).count()

null_customers = df.filter(col("Customer_ID").isNull()).count()

duplicates = df.groupBy("Order_ID","Product_ID").count().filter("count > 1").count()

negative_values = df.filter("Sales <= 0 OR Quantity <= 0").count()

invalid_discount = df.filter("Discount < 0 OR Discount > 1").count()

invalid_delivery = df.filter("delivery_days < 0").count()


from pyspark.sql import Row

report_data = [
    Row(check="total_rows", value=total_rows, status="INFO"),
    Row(check="null_order_id", value=null_orders, status="PASS" if null_orders == 0 else "FAIL"),
    Row(check="null_customer_id", value=null_customers, status="PASS" if null_customers == 0 else "FAIL"),
    Row(check="duplicate_orders", value=duplicates, status="PASS" if duplicates == 0 else "FAIL"),
    Row(check="negative_values", value=negative_values, status="PASS" if negative_values == 0 else "FAIL"),
    Row(check="invalid_discount", value=invalid_discount, status="PASS" if invalid_discount == 0 else "FAIL"),
    Row(check="invalid_delivery", value=invalid_delivery, status="PASS" if invalid_delivery == 0 else "FAIL")
]

report_df = spark.createDataFrame(report_data)


display(report_df)

In [0]:
report_df2.write.mode("overwrite").format("delta").saveAsTable("sales_project_db.data_quality_report_bronze")

In [0]:
report_df.write.mode("overwrite").format("delta").saveAsTable("sales_project_db.data_quality_report_silver")